# Module 1: Semantic Routing

This notebook demonstrates how to build a **Semantic Router** using LLMs. 
The router acts as a front door, intelligently classifying the complexity of an incoming receipt and routing it to the most cost-effective model.

### 1. Setup and Configuration
First, we import the necessary libraries and define our `RoutingConfig`. We use `gpt-4o-mini` for fast/cheap routing, and `gpt-4o` for high-precision extraction.

In [1]:
import os
import json
from pathlib import Path
from typing import Dict, Any, Optional
from dataclasses import dataclass
from openai import OpenAI

PROJECT_ROOT = Path(os.getcwd()).parent
env_path = f"{PROJECT_ROOT}/.env"

@dataclass
class RoutingConfig:
    """Configuration for model selection and API credentials."""
    # 'nano' is used for high-speed, low-cost classification (routing)
    router_model: str = "gpt-4o-mini"
    # 'mini' handles standard extractions with high efficiency
    simple_task_model: str = "gpt-4o-mini"
    # The flagship model is reserved for high-precision auditing tasks
    complex_task_model: str = "gpt-4o"
    openai_api_key: Optional[str] = None


### 2. The RouteLayer & Classification
The `RouteLayer` is responsible for initializing the OpenAI client and classifying the intent of the receipt. 

In the context of Large Language Models (LLMs) like OpenAI's GPT models, `temperature is a parameter that controls the randomness or creativity of the model's responses.` It typically ranges from 0.0 to 2.0 (though it's most commonly used between 0 and 1).

Here is how it works:
- 0.0 (what we are using here): Makes the model strictly focused, deterministic, and highly predictable. It will always choose the most probable next word.
- 0.5 - 0.7: A good balance of predictability and creativity. Often used for general chat or writing assistance.
- 1.0+: Very creative and unpredictable. The model might hallucinate or produce erratic outputs, which is sometimes useful for creative writing or brainstorming.

We use `temperature=0` here to ensure the classification decision is deterministic and consistent.

In [2]:
class RouteLayer:
    """
    A semantic router that directs receipts to different LLMs 
    based on complexity to balance cost and accuracy.
    """
    def __init__(self, config: Optional[RoutingConfig] = None) -> None:
        self.cfg = config or RoutingConfig()
        # Initialize client with provided key or fallback to environment variable
        self.client = OpenAI(api_key=self.cfg.openai_api_key or os.getenv("OPENAI_API_KEY"))

    def classify_intent(self, text: str) -> str:
        """
        Determines the complexity of the receipt to decide the processing path.
        Returns 'simple_meal' or 'complex_client_dinner'.
        """
        system_prompt = (
            "Classify this receipt into one of two categories:\n"
            "1. 'simple': Solo meals, coffee, fast food, under $30.\n"
            "2. 'complex': Group dinners, alcohol, steakhouse, clients, over $30.\n"
            "Respond with only the word 'simple' or 'complex'."
        )
        
        # We use temperature=0 for routing to ensure deterministic, consistent decisions
        response = self.client.chat.completions.create(
            model=self.cfg.router_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            temperature=0  
        )
        
        # Clean the response to ensure it matches our routing keys
        answer = response.choices[0].message.content.lower().strip()
        return "simple_meal" if "simple" in answer else "complex_client_dinner"
    def _extract_data(self, model: str, system_prompt: str, user_text: str) -> str:
        """
        Internal helper to perform the actual data extraction.
        Uses OpenAI's JSON mode to ensure valid dictionary outputs.
        """
        response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            # Ensures the model generates a string that can be parsed by json.loads()
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content

    def handle(self, receipt_text: str) -> Dict[str, Any]:
        """
        Entry point: Classifies the intent then routes to the specific prompt and model.
        """
        # Step 1: Decision phase
        intent = self.classify_intent(receipt_text)
        
        # Step 2: Routing phase
        if intent == "simple_meal":
            # Lightweight prompt for high-speed, low-cost processing
            system_prompt = "Extract JSON: {merchant, date, total, category: 'SimpleMeal'}"
            model = self.cfg.simple_task_model
        else:
            # Detailed prompt for expert auditing with risk detection
            system_prompt = (
                "You are an expert auditor. Extract JSON: "
                "{merchant, date, total, attendees, alcohol: bool, risk_flags: []}"
            )
            model = self.cfg.complex_task_model

        # Step 3: Execution phase
        output = self._extract_data(model, system_prompt, receipt_text)

        # Return a consolidated dictionary including metadata for tracking
        return {
            "intent": intent,
            "model_used": model,
            "data": json.loads(output) # Parse the raw string into a Python dict
        }


### 3. Testing the Setup
Let's test our `RouteLayer` with two receipts. Notice how the simple coffee receipt intelligently routes to the fast model, while the complex steakhouse dinner activates the expert model.

In [3]:
# --- Standard Testing Block ---
if __name__ == "__main__":
    router = RouteLayer()

    # Test Case 1: Minimal info, low cost
    receipt_simple = "Starbucks, 5.20 USD, latte, 2026-01-10"

    # Test Case 2: Multi-line, high cost, higher reasoning required
    receipt_complex = (
        "The Capital Grille\nTotal: 485.60 USD\n"
        "Hosted dinner with 4 clients, 2 bottles of wine\nDate: 2026-01-15"
    )

    print("\n--- Processing Simple (via GPT-4o-mini) ---")
    print(json.dumps(router.handle(receipt_simple), indent=2))

    print("\n--- Processing Complex (via GPT-4o) ---")
    print(json.dumps(router.handle(receipt_complex), indent=2))



--- Processing Simple (via GPT-4o-mini) ---
{
  "intent": "simple_meal",
  "model_used": "gpt-4o-mini",
  "data": {
    "merchant": "Starbucks",
    "date": "2026-01-10",
    "total": 5.2,
    "category": "SimpleMeal"
  }
}

--- Processing Complex (via GPT-4o) ---
{
  "intent": "complex_client_dinner",
  "model_used": "gpt-4o",
  "data": {
    "merchant": "The Capital Grille",
    "date": "2026-01-15",
    "total": 485.6,
    "attendees": 5,
    "alcohol": true,
    "risk_flags": []
  }
}
